# 03 Teacher training

3-layer, 128-unit GRU with dropout, trained with MSE on scaled targets. Checkpoint selection uses validation MSE only.

## Setup

In [1]:
# setup: clone repo, install deps, mount drive for saving models
import os, sys, subprocess
from google.colab import drive

REPO_URL = "https://github.com/BerkayPolat-lab/gru-currency-volatility-distillation.git"
REPO_DIR = "/content/gru-currency-volatility-distillation"

# clone fresh, or pull latest if already cloned this session
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
os.chdir(f"{REPO_DIR}/model")  # all paths in config relative to model/
sys.path.insert(0, f"{REPO_DIR}/model/src")  # lets notebooks do: from gru_vol import ...

# drive is where checkpoints live, so colab disconnect doesnt wipe trained models
drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/gru_vol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
print("ready, working dir:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready, working dir: /content/gru-currency-volatility-distillation/model


In [2]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)
CFG

{'data': {'source': 'FRED',
  'series_id': 'DEXUSEU',
  'start': '2000-01-01',
  'end': '2025-12-31',
  'snapshot_path': 'data/raw/DEXUSEU.csv',
  'interpolate_missing': False},
 'features': {'window': 60,
  'trailing_vol_windows': [5, 20],
  'inputs': ['log_return', 'squared_return', 'vol_5', 'vol_20'],
  'scaling': 'rolling_zscore',
  'scaling_window': 60},
 'target': {'horizon': 5, 'scaling': 'ratio_to_trailing_var'},
 'splits': {'train': ['2000-01-01', '2017-12-31'],
  'validation': ['2018-01-01', '2020-12-31'],
  'test': ['2021-01-01', '2025-12-31'],
  'drop_examples_crossing_split': True},
 'models': {'teacher': {'num_layers': 3, 'hidden_size': 128, 'dropout': 0.2},
  'student': {'num_layers': 1, 'hidden_size': 32, 'dropout': 0.0},
  'output_transform': 'softplus'},
 'historical_control': {'window': 60},
 'training': {'optimizer': 'adamw',
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'batch_size': 32,
  'max_epochs': 100,
  'early_stopping_metric': 'val_mse',
  'early_st

## Pilot

Run one full epoch to confirm peak GPU memory and per-epoch time before the full budget.

In [3]:
# load data + model, check gpu, time 1 epoch before full run
import json
import numpy as np
import torch
from gru_vol.models import build_model
from gru_vol.training import set_seed, train

device = "cuda" if torch.cuda.is_available() else "cpu"
npz = np.load("data/processed/splits.npz")
data = {k: npz[k] for k in npz.files if not k.startswith("dates_")}  # dates saved as objects, not needed for training
print("device:", device, "| train", data["X_train"].shape, "| val", data["X_validation"].shape)

# pilot: same config but only 1 epoch
pilot_cfg = {**CFG, "training": {**CFG["training"], "max_epochs": 1}}
set_seed(0)
pilot = train(build_model(CFG, "teacher"), data, pilot_cfg, seed=0, device=device)
print(f"1 epoch: {pilot['train_seconds']:.1f}s | val mse {pilot['best_val_mse']:.3e}")
if device == "cuda":
    print(f"peak gpu mem: {torch.cuda.max_memory_allocated() / 1e6:.0f} MB")

device: cuda | train (4437, 60, 4) | val (743, 60, 4)


epochs:   0%|          | 0/1 [00:00<?, ?it/s]

epoch   1 | train loss 0.7909 | val mse 2.5238e-10
1 epoch: 0.9s | val mse 2.524e-10
peak gpu mem: 172 MB


## Train per seed

AdamW, up to 100 epochs, early stopping on validation MSE. Log exact settings and wall-clock time.

In [6]:
# full teacher run, seed 0 only for midterm (3 seeds for final)
seed = 0
ckpt = f"artifacts/checkpoints/teacher_seed{seed}.pt"  # in repo so team gets it via git
os.makedirs(os.path.dirname(ckpt), exist_ok=True)

set_seed(seed)  # before build_model -> same init every rerun
teacher = build_model(CFG, "teacher")
res = train(teacher, data, CFG, seed=seed, device=device, ckpt_path=ckpt)

print(f"best val mse {res['best_val_mse']:.3e} at epoch {res['best_epoch']} | "
      f"ran {res['epochs_run']} epochs | {res['train_seconds']:.0f}s")

epochs:   0%|          | 0/100 [00:00<?, ?it/s]

best val mse 2.502e-10 at epoch 2 | ran 12 epochs | 5s


## Sanity check against the historical control

If the teacher is consistently worse than the smaller reference, reduce depth or report it. Never tune on test.

In [7]:
from gru_vol.training import predict
from gru_vol.evaluation import mse, high_vol_mse

# teacher forecast in original units: ratio * var_60
fc = predict(teacher, data["X_validation"], device) * data["var60_validation"]
hc = data["var60_validation"]  # hc forecast = ratio 1 = var_60
tgt = data["target_validation"]
cutoff = json.load(open("artifacts/manifests/data_manifest.json"))["target"]["high_vol_cutoff"]

hc_mse, t_mse = mse(tgt, hc), mse(tgt, fc)
(hc_hv, n_hv), (t_hv, _) = high_vol_mse(tgt, hc, cutoff), high_vol_mse(tgt, fc, cutoff)
print(f"val mse      hc {hc_mse:.3e} | teacher {t_mse:.3e} | {1 - t_mse / hc_mse:.1%} lower")
print(f"high vol mse hc {hc_hv:.3e} | teacher {t_hv:.3e} | n = {n_hv}")

# learning curve: train loss falling + val rising = overfitting
for h in res["history"]:
    print(f"epoch {h['epoch']:>3} | train loss {h['train_loss']:.4f} | val mse {h['val_mse']:.4e}")

# save results + checkpoint into repo so team gets them via git
os.makedirs("artifacts/results", exist_ok=True)
os.makedirs("artifacts/checkpoints", exist_ok=True)
torch.save(teacher.state_dict(), f"artifacts/checkpoints/teacher_seed{seed}.pt")  # best weights already loaded
with open(f"artifacts/results/teacher_seed{seed}.json", "w") as f:
    json.dump({**res, "hc_val_mse": hc_mse, "val_high_vol_mse": t_hv, "device": device}, f, indent=2)

val mse      hc 2.571e-10 | teacher 2.502e-10 | 2.7% lower
high vol mse hc 1.021e-08 | teacher 9.524e-09 | n = 8
epoch   1 | train loss 0.7927 | val mse 2.5266e-10
epoch   2 | train loss 0.7828 | val mse 2.5017e-10
epoch   3 | train loss 0.7833 | val mse 2.5432e-10
epoch   4 | train loss 0.7820 | val mse 2.5857e-10
epoch   5 | train loss 0.7773 | val mse 2.5425e-10
epoch   6 | train loss 0.7724 | val mse 2.9527e-10
epoch   7 | train loss 0.7646 | val mse 2.6532e-10
epoch   8 | train loss 0.7423 | val mse 2.7009e-10
epoch   9 | train loss 0.6894 | val mse 3.5406e-10
epoch  10 | train loss 0.6560 | val mse 2.7628e-10
epoch  11 | train loss 0.5569 | val mse 3.0139e-10
epoch  12 | train loss 0.4573 | val mse 4.1401e-10


In [4]:
# tuning grid on val only (test untouched), every config logged for report
import copy as cp
import itertools
import pandas as pd

grid = CFG["tuning"]["teacher"]  # learning_rate, dropout, weight_decay
hc_mse = float(np.mean((data["var60_validation"] - data["target_validation"]) ** 2))  # hc = ratio 1
seed, runs = 0, []

for lr, do, wd in itertools.product(grid["learning_rate"], grid["dropout"], grid["weight_decay"]):
    cfg = cp.deepcopy(CFG)  # fresh copy so base config never changes
    cfg["models"]["teacher"].update(learning_rate=lr, dropout=do, weight_decay=wd)
    print(f"\n--- lr {lr} | dropout {do} | wd {wd} ---")
    set_seed(seed)
    m = build_model(cfg, "teacher")
    r = train(m, data, cfg, seed=seed, device=device)
    runs.append({"lr": lr, "dropout": do, "weight_decay": wd, "val_mse": r["best_val_mse"],
                 "vs_hc": 1 - r["best_val_mse"] / hc_mse, "best_epoch": r["best_epoch"],
                 "epochs_run": r["epochs_run"], "seconds": round(r["train_seconds"])})

grid_df = pd.DataFrame(runs).sort_values("val_mse")
print(grid_df.to_string(index=False))


--- lr 0.001 | dropout 0.2 | wd 0.0001 ---


epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epoch   1 | train loss 0.7909 | val mse 2.5238e-10
epoch   2 | train loss 0.7836 | val mse 2.4965e-10
epoch   3 | train loss 0.7835 | val mse 2.5367e-10
epoch   4 | train loss 0.7822 | val mse 2.5855e-10
epoch   5 | train loss 0.7776 | val mse 2.5392e-10
epoch   6 | train loss 0.7728 | val mse 2.8764e-10
epoch   7 | train loss 0.7653 | val mse 2.6935e-10
epoch   8 | train loss 0.7360 | val mse 2.6662e-10
epoch   9 | train loss 0.6799 | val mse 3.4140e-10
epoch  10 | train loss 0.6003 | val mse 2.8418e-10
epoch  11 | train loss 0.4818 | val mse 2.9423e-10
epoch  12 | train loss 0.4202 | val mse 4.4072e-10

--- lr 0.001 | dropout 0.2 | wd 0.01 ---


epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epoch   1 | train loss 0.7909 | val mse 2.5236e-10
epoch   2 | train loss 0.7836 | val mse 2.4963e-10
epoch   3 | train loss 0.7835 | val mse 2.5362e-10
epoch   4 | train loss 0.7822 | val mse 2.5849e-10
epoch   5 | train loss 0.7777 | val mse 2.5379e-10
epoch   6 | train loss 0.7730 | val mse 2.8689e-10
epoch   7 | train loss 0.7659 | val mse 2.6762e-10
epoch   8 | train loss 0.7400 | val mse 2.6773e-10
epoch   9 | train loss 0.6864 | val mse 3.2727e-10
epoch  10 | train loss 0.5963 | val mse 3.0179e-10
epoch  11 | train loss 0.4856 | val mse 4.5532e-10
epoch  12 | train loss 0.4151 | val mse 4.8695e-10

--- lr 0.001 | dropout 0.4 | wd 0.0001 ---


...

epoch   1 | train loss 0.8016 | val mse 2.5014e-10
epoch   2 | train loss 0.7813 | val mse 2.4967e-10
epoch   3 | train loss 0.7829 | val mse 2.5028e-10
epoch   4 | train loss 0.7805 | val mse 2.4913e-10
epoch   5 | train loss 0.7807 | val mse 2.5175e-10
epoch   6 | train loss 0.7806 | val mse 2.5451e-10
epoch   7 | train loss 0.7811 | val mse 2.4726e-10
epoch   8 | train loss 0.7796 | val mse 2.4459e-10
epoch   9 | train loss 0.7805 | val mse 2.5538e-10
epoch  10 | train loss 0.7802 | val mse 2.5659e-10
epoch  11 | train loss 0.7811 | val mse 2.5101e-10
epoch  12 | train loss 0.7790 | val mse 2.5249e-10
epoch  13 | train loss 0.7786 | val mse 2.5536e-10
epoch  14 | train loss 0.7802 | val mse 2.4774e-10
epoch  15 | train loss 0.7777 | val mse 2.5101e-10
epoch  16 | train loss 0.7763 | val mse 2.5389e-10
epoch  17 | train loss 0.7766 | val mse 2.5188e-10
epoch  18 | train loss 0.7746 | val mse 2.6196e-10

--- lr 0.0001 | dropout 0.4 | wd 0.0001 ---


epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epoch   1 | train loss 0.8026 | val mse 2.5037e-10
epoch   2 | train loss 0.7819 | val mse 2.4919e-10
epoch   3 | train loss 0.7836 | val mse 2.4986e-10
epoch   4 | train loss 0.7808 | val mse 2.4955e-10
epoch   5 | train loss 0.7820 | val mse 2.5168e-10
epoch   6 | train loss 0.7809 | val mse 2.5449e-10
epoch   7 | train loss 0.7814 | val mse 2.4750e-10
epoch   8 | train loss 0.7797 | val mse 2.4482e-10
epoch   9 | train loss 0.7813 | val mse 2.5485e-10
epoch  10 | train loss 0.7805 | val mse 2.5638e-10
epoch  11 | train loss 0.7820 | val mse 2.5038e-10
epoch  12 | train loss 0.7806 | val mse 2.5181e-10
epoch  13 | train loss 0.7784 | val mse 2.5443e-10
epoch  14 | train loss 0.7816 | val mse 2.4646e-10
epoch  15 | train loss 0.7792 | val mse 2.4948e-10
epoch  16 | train loss 0.7783 | val mse 2.5263e-10
epoch  17 | train loss 0.7779 | val mse 2.4908e-10
epoch  18 | train loss 0.7762 | val mse 2.5739e-10

--- lr 0.0001 | dropout 0.4 | wd 0.01 ---


epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epoch   1 | train loss 0.8026 | val mse 2.5037e-10
epoch   2 | train loss 0.7819 | val mse 2.4919e-10
epoch   3 | train loss 0.7836 | val mse 2.4985e-10
epoch   4 | train loss 0.7808 | val mse 2.4955e-10
epoch   5 | train loss 0.7820 | val mse 2.5168e-10
epoch   6 | train loss 0.7809 | val mse 2.5448e-10
epoch   7 | train loss 0.7814 | val mse 2.4750e-10
epoch   8 | train loss 0.7798 | val mse 2.4482e-10
epoch   9 | train loss 0.7813 | val mse 2.5483e-10
epoch  10 | train loss 0.7805 | val mse 2.5637e-10
epoch  11 | train loss 0.7820 | val mse 2.5036e-10
epoch  12 | train loss 0.7806 | val mse 2.5180e-10
epoch  13 | train loss 0.7784 | val mse 2.5442e-10
epoch  14 | train loss 0.7816 | val mse 2.4644e-10
epoch  15 | train loss 0.7792 | val mse 2.4946e-10
epoch  16 | train loss 0.7783 | val mse 2.5261e-10
epoch  17 | train loss 0.7780 | val mse 2.4905e-10
epoch  18 | train loss 0.7763 | val mse 2.5731e-10
    lr  dropout  weight_decay      val_mse    vs_hc  best_epoch  epochs_run  secon

In [6]:
# best config across 3 seeds + intuitive metrics
from gru_vol.training import predict
# the grid winner must already be copied into models.teacher in base.yaml
winner, teacher_cfg = grid_df.iloc[0], CFG["models"]["teacher"]
assert np.isclose(teacher_cfg["learning_rate"], winner["lr"]) and np.isclose(teacher_cfg["dropout"], winner["dropout"]), \
    f"copy the grid winner (lr {winner['lr']}, dropout {winner['dropout']}) into models.teacher in base.yaml"
best_cfg = CFG

y_val, tgt, v60 = data["y_validation"], data["target_validation"], data["var60_validation"]
always_lower = np.mean(y_val < 1)  # naive call: "next week calmer than last 60 days" every day

def r2(f):
    """share of realized vol ups/downs explained by forecast (r2 = corr^2)"""
    return np.corrcoef(f, tgt)[0, 1] ** 2

print(f"hc: mse {hc_mse:.3e} | r2 {r2(v60):.3f} | 'always calmer' hit rate {always_lower:.1%}")
os.makedirs("artifacts/checkpoints", exist_ok=True)
os.makedirs("artifacts/results", exist_ok=True)
rows = []
for s in [0, 1, 2]:
    set_seed(s)
    m = build_model(best_cfg, "teacher")
    r = train(m, data, best_cfg, seed=s, device=device, verbose=False)
    ratio = predict(m, data["X_validation"], device)
    rows.append({"seed": s, "val_mse": r["best_val_mse"], "vs_hc": 1 - r["best_val_mse"] / hc_mse,
                 "hit_rate": np.mean((ratio > 1) == (y_val > 1)), "r2": r2(ratio * v60),
                 "best_epoch": r["best_epoch"]})
    torch.save(m.state_dict(), f"artifacts/checkpoints/teacher_seed{s}.pt")

seeds_df = pd.DataFrame(rows)
seeds_df.to_csv("artifacts/results/teacher_seeds.csv", index=False)
grid_df.to_csv("artifacts/results/teacher_grid.csv", index=False)
print(seeds_df.to_string(index=False))
print(f"mean vs hc {seeds_df.vs_hc.mean():.1%} ± {seeds_df.vs_hc.std():.1%}")

hc: mse 2.571e-10 | r2 0.068 | 'always calmer' hit rate 62.6%


epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epochs:   0%|          | 0/100 [00:00<?, ?it/s]

 seed      val_mse    vs_hc  hit_rate       r2  best_epoch
    0 2.437513e-10 0.051826  0.551817 0.098304           8
    1 2.453082e-10 0.045770  0.541050 0.099167           3
    2 2.460341e-10 0.042946  0.546433 0.100508           6
mean vs hc 4.7% ± 0.5%
